# Exploratory Data Analysis

Packages

In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import math
import os
import re

# Custom packages
from filter import FilterDF as fdf
from benchmarks import ParetoAnalysis as pa
from benchmarks import AccuracyCalculation as ac
from integrity_fixes import DataFixer as fix, DataExporter as exporter

In [ ]:
pd.options.mode.copy_on_write = True

In [ ]:
%load_ext autoreload
%autoreload 2

Load formatted data

In [ ]:
%store -r static_data_merged
%store -r sales_data_merged

Read data from parquet files

In [ ]:
# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    %store static_data_merged


# Data already exists
else:
    static_data = static_data_merged.copy()

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
    %store sales_data_merged

# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()


# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

# Remake into string variable
items_tagged['dish_category'] = items_tagged['dish_category'].astype(str)

## Static Reference Data Exploration

### Menu Stats

In [ ]:
items_tagged.columns.tolist()

Specific

In [ ]:
items_tagged['dish_category'].where()

In [ ]:
# How many unique items must it have to be considered a useful category?
unique_items = 3
dish_category_counts = items_tagged['dish_category'].value_counts().nlargest(20)
usable_categories = dish_category_counts[dish_category_counts > unique_items].index.tolist()

# Throw these smaller categories into a larger 'Other' category
dish_categories_other = items_tagged['dish_category'].where(items_tagged['dish_category'].isin(usable_categories), 'Other')
items_tagged = items_tagged.assign(dish_category=dish_categories_other)

### Menu Visuals

In [ ]:
# Copy for visualization processing
items_tagged_copy = items_tagged.copy()

# How many categories to include?
topn = 20
top_categories = items_tagged_copy['dish_category'].value_counts().nlargest(topn).index.tolist()

# Throw these smaller categories into a larger 'Other' category, but purely for visualization
dish_categories_other = items_tagged_copy['dish_category'].where(items_tagged_copy['dish_category'].isin(top_categories), 'Other')
items_tagged_copy = items_tagged_copy.assign(dish_category=dish_categories_other)

#

# Add an 'included_pb' column to differentiate plant-based entries
items_tagged_copy['included_pb'] = 'all'
plant_based = items_tagged_copy.query('is_plant_based == "yes"')
plant_based = plant_based.assign(included_pb = 'yes')

# Add an 'included_alc' column to differentiate alcohol entries
items_tagged_copy['included_alc'] = 'all'
no_alcohol = items_tagged_copy.query('dish_category != "Alcohol"')
no_alcohol = no_alcohol.assign(included_alc= 'no alcohol')

# Intersection of the two
plant_based_no_alcohol = plant_based.query('dish_category != "Alcohol"')
plant_based_no_alcohol = plant_based_no_alcohol.assign(included_alc= 'no alcohol')

# Duplicate the modified rows to separate them in the visualizations
items_tagged_full = pd.concat([items_tagged_copy, plant_based, no_alcohol, plant_based_no_alcohol])

In [ ]:
items_tagged_copy['is_alcohol'] = items_tagged_copy['item_type'] == 'Alcohol'

In [ ]:
sns.catplot(
    data=items_tagged_copy, 
    x='is_plant_based', 
    col='is_alcohol',
    #hue='included_pb',
    #palette='mako',
    kind='count',
    order=items_tagged['is_plant_based'].value_counts().index
)

In [ ]:
sns.catplot(
    data=items_tagged_full, 
    x='item_type', 
    col='included_alc',
    row='included_pb',
    hue='included_pb',
    palette='mako',
    kind='count',
    order=items_tagged['item_type'].value_counts().index
)

In [ ]:
sns.catplot(
    data=items_tagged_full, 
    x='item_type', 
    col='included_alc',
    row='included_pb',
    hue='included_pb',
    palette='mako',
    kind='count',
    order=items_tagged['item_type'].value_counts().index
)

In [ ]:
plot = sns.catplot(
    data=items_tagged_full, 
    x='dish_category', 
    col='included_alc',
    row='included_pb',
    hue='included_pb',
    palette="viridis",
    kind='count',
    order=items_tagged_copy['dish_category'].value_counts().index
)

plot.set_xticklabels(rotation=90)
plot.tight_layout()

plt.show()

In [ ]:
plt.setp(g.ax.get_xticklabels(), rotation=90)

# Adjust layout
plt.tight_layout()

In [ ]:
sns.catplot(
    data=items_tagged, 
    x='item_type', y=, 
    col='is_alcohol', row='is_plant_based'
    kind="bar", height=4, aspect=.6,
)

In [ ]:
grid = sns.FacetGrid(items_tagged, col='is_alcohol', row='is_plant_based')

# Define the custom plotting function
def custom_plot(x, y, **kwargs):
    if kwargs.get('subset_condition', False):
        subset_data = items_tagged[kwargs['subset_condition']]
        sns.barplot(data=subset_data, x=x, y=y, ci=None, **kwargs)
    else:
        sns.stripplot(data=items_tagged, x=x, y=y, jitter=True, ax=plt.gca(), **kwargs)

# Map the custom plotting function to the grid
grid.map_dataframe(custom_plot, x='item_type', y='your_y_variable', subset_condition={'is_alcohol': True})

In [ ]:
grid = sns.FacetGrid(items_tagged, col='is_alcohol', row='is_plant_based')
grid.map_dataframe(sns.histplot, x="item_type")

In [ ]:
grid = sns.FacetGrid(items_tagged, col='is_alcohol', row='is_plant_based')
grid.map(sns.barplot, x='dish_category')

In [ ]:
plt.bar(items_tagged['is_plant_based'].value_counts().index, items_tagged['is_plant_based'].value_counts())
plt.title("Total Menu Items: Is it Plant-Based?")
plt.show()

## Restaurant Sales Data Exploration

Timeframes

In [ ]:
list_of_timeframes = []

# Find the time difference
for location_id, df in tqdm(sales_and_menu_data.items()):

    # Find the time difference
    timedelta = df.index[-1] - df.index[0]

    # Convert to days, then to years
    years = timedelta.days / 365.25

    # Append
    list_of_timeframes.append(years)

# Turn into an np array for finding median, mean, and std
timeframes = np.array(list_of_timeframes)

# Display
print("Median: {:.2f} year range".format(np.median(timeframes)))
print("Mean: {:.2f} year range".format(np.mean(timeframes)))
print("SD: {:.2f} years".format(np.std(timeframes)))
print("Restaurants with less than a 2 year range: {}".format((timeframes < 2).sum()))

## Sales Data Coverage

Totals

In [ ]:
# Recheck total number of entries
sales_total = 0
for loc_id, df in sales_and_menu_data.items():
    sales_total += df.shape[0]
sales_total

Timing

In [ ]:
## Find the overall date range

# # Start
# start_dates = []
# for loc, df in sales_and_menu_data.items():
#     start_dates.append(df.index.min())
# first_start_date = pd.Series(start_dates).sort_values().iloc[0]
# first_start_date = first_start_date.tz_localize(None) # Remove timezone info

# # End
# end_date = pd.to_datetime("now") # No timezone info because we're converting to weeks

# # Generate a complete range of weeks from start to end
# all_weeks = pd.date_range(start=first_start_date, end=end_date, freq='W').to_period('W')

# Calculate active weeks
active_weeks_dict = {}
for loc_id, df in sales_and_menu_data.items():
    df = df.copy()
    df.index = df.index.tz_localize(None) # Remove timezone info
    df['Week'] = df.index.to_period('W')
    active_weeks_dict[loc_id] = set(df.groupby('Week').size().index.tolist())

### Coverage Visual

In [ ]:
# Visualizing with gaps for inactive weeks
plt.figure(figsize=(14, 8))

# Loop through every active week
for base_name, active_weeks in active_weeks_dict.items(): # active_weeks is all the active weeks in a single restaurant

    # Change to restaurant ID
    loc_id = re.sub(r'_sales_and_menu$', '', base_name)

    # For every active week
    for week in active_weeks:

        # Place a blue dot
        plt.hlines(y=loc_id, xmin=week.start_time, xmax=week.end_time, colors='blue', lw=2, label=loc_id)

    # Index into the promotional items for this restaurant
    promo_datetime = before_after_details.loc[loc_id,'cross_over_date']

    # Place a red circle for the promotional item
    plt.plot(promo_datetime, loc_id, 'ro', alpha=0.5)

# Plot
plt.title('Weekly Activity for Each Restaurant with Gaps for Inactive Weeks')
plt.xlabel('Date')
plt.ylabel('Restaurant ID')
plt.yticks()
plt.tight_layout()
plt.show()

### Data Density

In [ ]:
# Initialize a list to see if there is a data buffer before and after the promotional item introduction
data_density_list = []
for loc_id, df in list(sales_and_menu_data.items()):
    
    # Copy to prevent removal of timezone
    df = df.copy()
    df.index = df.index.tz_localize(None)

    # Identify introductin date
    promo_datetime = before_after_details.loc[loc_id, 'cross_over_date']

    half_day_counts = df['item_name'].resample('12H').count()
    daily_counts = df['item_name'].resample('D').count()
    weekly_counts = df['item_name'].resample('W-MON').count()
    monthly_counts = df['item_name'].resample('M').count()

    # first_day = half_day_counts.index[0]
    # last_day = half_day_counts.index[half_day_counts.shape[0]-1]

    half_day_coverage_ratio = round(100 * (half_day_counts > 1).sum() / half_day_counts.size)/100
    half_day_mean = round(np.mean(half_day_counts))
    half_day_sd = round(np.std(half_day_counts)) 

    daily_coverage_ratio = round(100 * (daily_counts > 1).sum() / daily_counts.size)/100
    daily_mean = round(np.mean(daily_counts))
    daily_sd = round(np.std(daily_counts))

    weekly_coverage_ratio = round(100 * (weekly_counts > 1).sum() / weekly_counts.size)/100
    weekly_mean = round(np.mean(weekly_counts))
    weekly_sd = round(np.std(weekly_counts))

    monthly_coverage_ratio = round(100 * (monthly_counts > 1).sum() / monthly_counts.size)/100
    monthly_mean = round(np.mean(monthly_counts))
    monthly_sd = round(np.std(monthly_counts))

    # Aggregate for summary
    row = {'loc_id': loc_id,
           'hd_coverage': half_day_coverage_ratio,
           'd_coverage': daily_coverage_ratio,
           'w_coverage': weekly_coverage_ratio,
           'm_coverage': monthly_coverage_ratio,
           'hd_mean': half_day_mean, 
           'hd_sd': half_day_sd, 
           'd_mean': daily_mean, 
           'd_sd': daily_sd, 
           'w_mean': weekly_mean, 
           'w_sd': weekly_sd,
           'm_mean': monthly_mean, 
           'm_sd': monthly_sd, }
    
    data_density_list.append(row)

data_density = pd.DataFrame(data_density_list)

In [ ]:
data_density.sort_values('hd_coverage')

### Buffer Data Before and After Promo

In [ ]:
# Initialize a list to see if there is a data buffer before and after the promotional item introduction
buffer_data_list = []
for loc_id, df in sales_and_menu_data.items():
    
    # Copy to prevent removal of timezone
    df = df.copy()
    df.index = df.index.tz_localize(None)

    # Identify introductin date
    promo_datetime = before_after_details.loc[loc_id, 'cross_over_date']

    # Create an offset because we want data two months before and after the promotional introduction date
    before_limit = promo_datetime - pd.DateOffset(months=2)
    after_limit = promo_datetime + pd.DateOffset(months=2)

    # Slice to before and after
    before_data = df.loc[:promo_datetime]
    after_data = df.loc[promo_datetime:]

    # Bound by two months
    two_months_prior = before_data.loc[before_limit:]
    two_months_after = after_data.loc[:after_limit]

    # Number of data entries before and after
    before_data_count = before_data.shape[0]
    after_data_count = after_data.shape[0]

    # Number of active weeks before and after
    before_week_count = (0 < before_data['item_name'].resample('W-MON').count()).sum()
    after_week_count = (0 < after_data['item_name'].resample('W-MON').count()).sum()

    # Number of data entries within the bounds
    two_months_prior_data_count = two_months_prior.shape[0]
    two_months_after_data_count = two_months_after.shape[0]

    # Number of active weeks within the bounds
    two_months_prior_week_count = (0 < two_months_prior['item_name'].resample('W-MON').count()).sum()
    two_months_after_week_count = (0 < two_months_after['item_name'].resample('W-MON').count()).sum()

    # Data coverage within the bounds
    two_months_prior_half_day_counts = before_data['item_name'].resample('12H').count()
    two_months_prior_half_day_coverage_ratio = round(100 * (two_months_prior_half_day_counts > 1).sum() / two_months_prior_half_day_counts.size)/100
    two_months_after_half_day_counts = before_data['item_name'].resample('12H').count()
    two_months_after_half_day_coverage_ratio = round(100 * (two_months_after_half_day_counts > 1).sum() / two_months_after_half_day_counts.size)/100

    # Aggregate for summary
    row = {'loc_id': loc_id, 
           'b_entries': before_data_count, 
           'b_weeks': before_week_count, 
           'b2m_entries': two_months_prior_data_count,
           'b2m_weeks': two_months_prior_week_count,
           'b2m_hd_coverage': two_months_prior_half_day_coverage_ratio,
           'a_entries': after_data_count, 
           'a_weeks': after_week_count, 
           'a2m_entries': two_months_after_data_count,
           'a2m_weeks': two_months_after_week_count,
           'a2m_hd_coverage': two_months_after_half_day_coverage_ratio}
    buffer_data_list.append(row)
    
buffer_data = pd.DataFrame(buffer_data_list)

In [ ]:
buffer_data.sort_values('b2m_hd_coverage')

In [ ]:
# Don't overwrite
df = sales_and_customers_data['2HRX9P6HKXA8V'].copy()

df['year'] = df['created_at'].dt.year
df['month'] = df['created_at'].dt.month
df['week'] = df['created_at'].dt.isocalendar().week
df['day'] = df['created_at'].dt.day

df['plant_based_sales'] = df.apply(lambda x: x['item_price'] if x['is_plant_based'] == 'yes' else 0, axis=1)
df['plant_based_quantity'] = df.apply(lambda x: x['item_quantity'] if x['is_plant_based'] == 'yes' else 0, axis=1)

aggregated_data = df.groupby(['location_id', 'year', 'month', 'week', 'day']).agg(
    transactions=('order_id', 'nunique'),
    all_sales=('item_price', 'sum'),
    plant_based_sales=('plant_based_sales', 'sum'),
    all_items=('item_quantity', 'sum'),
    plant_based_items=('plant_based_quantity', 'sum'),
    percent_female=('gender', lambda x: (x == 'female').mean() * 100),
).reset_index()

aggregated_data['percent_items_plant_based'] = (aggregated_data['plant_based_items'] / aggregated_data['all_items']) * 100
aggregated_data['percent_sales_plant_based'] = (aggregated_data['plant_based_sales'] / aggregated_data['all_sales']) * 100

aggregated_data

In [ ]:
sales_and_customers_data['2HRX9P6HKXA8V'].head(5)

In [ ]:
# Don't overwrite
df = sales_and_customers_data['2HRX9P6HKXA8V'].copy()

# Resample the DataFrame by week
weekly_data = df.resample('W')

# Calculate metrics for plant-based
plant_based_counts = weekly_data['is_plant_based'].apply(lambda x: (x == 'yes').sum())
total_entries = weekly_data.size()
plant_based_percentage = round((plant_based_counts / total_entries) * 100)

# Calculate metrics for gender
female_counts = weekly_data['gender'].apply(lambda x: (x == 'female').sum())
known_gender_counts = weekly_data['gender'].apply(lambda x: x.notna().sum())
female_percentage = round((female_counts / known_gender_counts) * 100)

# Combine all metrics into a single DataFrame
weekly_df = pd.DataFrame({
    'Plant-Based Counts': plant_based_counts,
    'Plant-Based Percentage': plant_based_percentage,
    'Female Counts': female_counts,
    'Female Percentage': female_percentage.fillna(0)  # fill NaN with 0 where gender is unknown
})

In [ ]:
weekly_data_list = []
for loc_id, df in sales_and_customers_data.items():

    df = df.copy()

    df['year'] = df['created_at'].dt.year
    df['month'] = df['created_at'].dt.month
    df['week'] = df['created_at'].dt.isocalendar().week
    df['day'] = df['created_at'].dt.day

    df['plant_based_sales'] = df.apply(lambda x: x['item_price'] if x['is_plant_based'] == 'yes' else 0, axis=1)
    df['plant_based_quantity'] = df.apply(lambda x: x['item_quantity'] if x['is_plant_based'] == 'yes' else 0, axis=1)

    aggregated_data = df.groupby(['location_id', 'year', 'month', 'week', 'day']).agg(
        transactions=('order_id', 'nunique'),
        all_sales=('item_price', 'sum'),
        plant_based_sales=('plant_based_sales', 'sum'),
        all_items=('item_quantity', 'sum'),
        plant_based_items=('plant_based_quantity', 'sum'),
        percent_female=('gender', lambda x: (x == 'female').mean() * 100),
    ).reset_index()

    aggregated_data['percent_items_plant_based'] = (aggregated_data['plant_based_items'] / aggregated_data['all_items']) * 100
    aggregated_data['percent_sales_plant_based'] = (aggregated_data['plant_based_sales'] / aggregated_data['all_sales']) * 100

    weekly_data_list.append(aggregated_data)

In [ ]:
weekly_data = pd.concat(weekly_data_list)

weekly_data_for_export = pd.merge(weekly_data, locations, on='location_id', how='left')

In [ ]:
weekly_data_for_export.columns

In [ ]:
weekly_data_for_export

In [ ]:
weekly_data_for_export.to_parquet('weekly_data.parquet')

## Customer Data Exploration

In [ ]:
# Number of customers, number of customers without age data, number of customers without gender data
print(customers.shape[0], customers['age'].isna().sum(), customers['gender'].isna().sum())

In [ ]:
precomputed_customers = {}
for loc_id in location_ids:
    precomputed_customers[loc_id] = set(fdf(customers).filter('location_id', loc_id)['customer_id'].unique().tolist())

In [ ]:
for i in range(len(location_ids)):
    loc_id1 = location_ids[i]
    customer_set1 = precomputed_customers[loc_id1]
    for loc_id2 in location_ids[i:]:
        if loc_id1 != loc_id2:
            customer_set2 = precomputed_customers[loc_id2]
            intersection = customer_set1.intersection(customer_set2)
            if intersection:
                print(loc_id1, loc_id2, len(intersection))
            else:
                pass
                # print(loc_id1, loc_id2, "Nope")

In [ ]:
# Initialize dict all data
sales_and_customers_data = {}
for loc_id, df in sales_and_menu_data.items():

    # Prevent overwriting
    df = df.copy()

    # Keep the index, since merges don't keep it
    df.reset_index(inplace=True)

    # Double check customers are unique
    customers.dropna(subset=['customer_id'], inplace=True)
    customers.drop_duplicates(subset=['location_id', 'customer_id'], inplace=True)

    # Combine
    merged = pd.merge(df, customers, on=['location_id', 'customer_id'], how='left')

    # All entries have a batch, since it was adding after data processing
    # print(merged[~merged['customer_id'].isna() & merged['batch'].isna()].size)
    # print(merged[merged['batch'].isna()].size)

    # Reset the index back to datetimes
    merged.set_index('created_at', inplace=True, drop=False)

    # Save
    sales_and_customers_data[loc_id] = merged

In [ ]:
sales_and_customers['0RJH3FFPYBPEY'].columns()

Totals

In [ ]:
combined_total = 0
for loc_id, df, in sales_and_customers_data.items():
    combined_total += df.shape[0]
combined_total

In [ ]:
female_proportions_list = []

for loc_id in location_ids:

    row = {'location_id': loc_id}

    cross_over = before_after_details.loc[loc_id]['cross_over_date']

    before_genders = merged_sales_and_customers[loc_id].loc[:cross_over]['gender'].value_counts()
    after_genders = merged_sales_and_customers[loc_id].loc[cross_over:]['gender'].value_counts()
    
    if not before_genders.empty and not after_genders.empty:

        before_female_total = before_genders.loc['female']
        after_female_total = after_genders.loc['female']
        before_known_gender_total = before_genders.loc['male'] + before_genders.loc['female']
        after_known_gender_total = after_genders.loc['male'] + after_genders.loc['female']

        before_frac_female = -1
        if before_known_gender_total != 0:
            before_frac_female = before_female_total/before_known_gender_total
            
        after_frac_female = -1
        if after_known_gender_total != 0:
            after_frac_female = after_female_total/after_known_gender_total
            
        sample_qualifer = ""
        if 1000 < before_known_gender_total and 1000 < after_known_gender_total:
            sample_qualifer = "Big Enough Sample"

        # print(loc_id, round(before_frac_female*100)/100, round(after_frac_female*100)/100, before_known_gender_total, after_known_gender_total, sample_qualifer)

        row['female_proportion_before'] = round(before_frac_female*100)/100
        row['female_proportion_after'] = round(after_frac_female*100)/100
        row['enough_data'] = bool(sample_qualifer)
        row['before_known_gender_total'] = before_known_gender_total
        row['after_known_gender_total'] = after_known_gender_total

    elif before_genders.empty and after_genders.empty:

        print(loc_id, "--No customer data!")

    else:

        print(loc_id, "--Not enough data before or after.")

    female_proportions_list.append(row)



female_proportions = pd.DataFrame(female_proportions_list)
female_proportions

In [ ]:
# fdf(items_tagged).filter('item_name', '16 monkey boy')
# fdf(merged_sales_and_menu['EMBVNVD207CC6']).filter('item_name', '16 monkey boy')
# fdf(merged_sales_and_menu['EMBVNVD207CC6']).filter('dish_category', 'alcohol', exclude=True)['item_name'].value_counts()

In [ ]:
customer_revisit_row_list = []
for loc_id, df in sales_and_menu_data.items():
    
    row = {'location_id': loc_id, 'total': df['customer_id'].nunique()}

    for j in [1, 2, 5, 10]:

        merge_successes = df[~df['batch'].isna()]
        customers_revisits_j = merge_successes.groupby('customer_id')['created_at'].nunique() > j
        num_customer_revisits = customers_revisits_j.sum()
        row['more than ' + str(j)] = num_customer_revisits

    customer_revisit_row_list.append(row)

revisits = pd.DataFrame(customer_revisit_row_list)

## Promotional Items

In [ ]:
promo_match_list = []

for loc_id, df in merged_sales_and_menu.items():

    # Looking for promo
    promo_item = before_after_details.loc[loc_id, 'first_plant_based_mention']
    cross_over_date = before_after_details.loc[loc_id, 'cross_over_date']
    promo_df = fdf(df).filter('item_name', promo_item)
    ever_found = not promo_df.empty


    # Actual first
    plant_based = fdf(df).filter('is_plant_based', 'yes')
    first_plant_based = plant_based['item_name'].iloc[0]
    its_date = plant_based.index[0]

    # Do they match?
    is_first = promo_item.lower() == first_plant_based.lower()

    row = {'location_id': loc_id, 'promo_item': promo_item, 'cross_over_date': cross_over_date, 'ever_found': ever_found, 'is_first': is_first, 'first_plant_based': first_plant_based, 'its_date': its_date}

    promo_match_list.append(row)
    
pd.DataFrame(promo_match_list)

## Metrics

In [ ]:
merged_sales_and_menu[location_ids[0]].resample('M')['item_name'].count().head(50)

In [ ]:
plt.plot(fdf(merged_sales_and_menu[location_ids[0]]).filter('is_plant_based','yes').resample('M')['item_name'].count() / merged_sales_and_menu[location_ids[0]].resample('M')['item_name'].count())

In [ ]:
weekly_pb_list = []
monthly_pb_list = []
yearly_pb_list = []

visual_dict = {}

for loc_id, df in merged_sales_and_menu.items():

    fig, ax = plt.subplots()  # Creates a single subplot


    # df['week'] = df.index.isocalendar().week
    # df['month'] = df.index.month
    # df['year'] = df.index.year
    plant_based = fdf(df).filter('is_plant_based','yes')
    promo_datetime = pd.to_datetime(before_after_details.loc[loc_id]['cross_over_date'])
    # weekly_pb_list.append(plant_based.groupby('week')['item_name'].count())
    # monthly_pb_list.append(plant_based.groupby('month')['item_name'].count())
    # yearly_pb_list.append(plant_based.groupby('year')['item_name'].count())
    print(loc_id)
    ax.plot(plant_based.resample('M')['item_name'].count() / df.resample('M')['item_name'].count())
    ax.axvline(x=promo_datetime, color='red', linestyle='--')
    ax.set_title(loc_id)
    visual_dict[loc_id] = ax


## Sales Visuals

In [ ]:
num_plots = len(sales_and_menu_data)
cols = 4 
rows = math.ceil(num_plots / cols) * 4

# Create a figure with multiple subplots
fig, axs = plt.subplots(rows, cols, figsize=(15, 5 * rows))
axs = axs.flatten()  # Flatten the array for easy indexing

top_dishes_number = 20

for i, (location_id, data) in tqdm(enumerate(sales_and_menu_data.items())):

    df = data.copy()
    
    # Get the top 20 items sorted in descending order
    top_items_by_times_ordered = df['item_name'].value_counts().nlargest(top_dishes_number).sort_values()

    # Plot in the appropriate subplot as a horizontal bar chart
    axs[4*i].barh(top_items_by_times_ordered.index.str.slice(0,20).str.capitalize(), top_items_by_times_ordered)
    axs[4*i].set_title(f'{location_id}\nTotal by Times Ordered')
    
    # Set y-axis label and make y-labels smaller
    axs[4*i].tick_params(axis='y', labelsize=10)
    axs[4*i].set_xlabel('Times Ordered')

    top_items_by_quantity_ordered = df.groupby('item_name')['item_quantity'].sum().nlargest(top_dishes_number).sort_values()
    
    # Plot in the appropriate subplot as a horizontal bar chart
    axs[4*i + 1].barh(top_items_by_quantity_ordered.index.str.slice(0,20).str.capitalize(), top_items_by_quantity_ordered, color='cyan')
    axs[4*i + 1].set_title(f'{location_id}\nTotal by Quantity Ordered')
    
    # Set y-axis label and make y-labels smaller
    axs[4*i + 1].tick_params(axis='y', labelsize=10)
    axs[4*i + 1].set_xlabel('Quantity Ordered')

    plant_df = df[df['is_plant_based'] == 'yes']
    
    # Get the top 20 items sorted in descending order
    top_plant_based_items_by_times_ordered = plant_df['item_name'].value_counts().nlargest(top_dishes_number).sort_values()

    # Plot in the appropriate subplot as a horizontal bar chart
    axs[4*i + 2].barh(top_plant_based_items_by_times_ordered.index.str.slice(0,20).str.capitalize(), top_plant_based_items_by_times_ordered, color='green')
    axs[4*i + 2].set_title(f'{location_id}\nPlant-Based by Times Ordered')
    
    # Set y-axis label and make y-labels smaller
    axs[4*i + 2].tick_params(axis='y', labelsize=10)
    axs[4*i + 2].set_xlabel('Times Ordered')

    top_plant_based_items_by_quantity_ordered = plant_df.groupby('item_name')['item_quantity'].sum().nlargest(top_dishes_number).sort_values()
    
    # Plot in the appropriate subplot as a horizontal bar chart
    axs[4*i + 3].barh(top_plant_based_items_by_quantity_ordered.index.str.slice(0,20).str.capitalize(), top_plant_based_items_by_quantity_ordered, color='lime')
    axs[4*i + 3].set_title(f'{location_id}\nPlant-Based by Quantity Ordered')
    
    # Set y-axis label and make y-labels smaller
    axs[4*i + 3].tick_params(axis='y', labelsize=10)
    axs[4*i + 3].set_xlabel('Quantity Ordered')

# # Add a global title at the top of the figure
# fig.suptitle('Distribution of Plant-Based and Total Item Orders in Each Restaurant', fontsize=16)

# # Adjust layout and save the figure
# plt.tight_layout()
# plt.subplots_adjust(top=0.95)  # Adjust the top margin to make room for the global title
# plt.savefig('Restaurant Both Sales Distributions.png', bbox_inches='tight')